<a href="https://colab.research.google.com/github/DikshantBadawadagi/Qwen-2.5-7B-local/blob/main/Local_Qwen2_5_3B_Instruct_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install transformers bitsandbytes accelerate flask flask-ngrok pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import gc, torch

for var in ['model', 'tokenizer']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
print(f"✅ Model loaded! VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Mounted at /content/drive
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded! VRAM used: 1.91 GB


In [ ]:
import torch, time, json, asyncio
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from pyngrok import ngrok
import nest_asyncio, uvicorn

nest_asyncio.apply()

app = FastAPI()
MODEL_NAME = "qwen2.5-3b-instruct"

def run_generation(messages, max_tokens=512, temperature=0.7):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# ── Open WebUI hits this first to check model list ───────────────────────────
@app.get("/api/tags")
async def list_models():
    return {
        "models": [{
            "name": MODEL_NAME,
            "model": MODEL_NAME,
            "modified_at": "2025-01-01T00:00:00Z",
            "size": 3000000000,
            "digest": "sha256:abc123",
            "details": {
                "format": "gguf",
                "family": "qwen",
                "parameter_size": "3B",
                "quantization_level": "Q4_K_M"
            }
        }]
    }


# ── Open WebUI hits this for model metadata ───────────────────────────────────
@app.post("/api/show")
async def show_model(request: Request):
    return {
        "modelfile": "",
        "parameters": "num_ctx 4096",
        "template": "",
        "details": {
            "format": "gguf",
            "family": "qwen",
            "parameter_size": "3B",
            "quantization_level": "Q4_K_M"
        }
    }


# ── Main chat endpoint ────────────────────────────────────────────────────────
@app.post("/api/chat")
async def chat(request: Request):
    data        = await request.json()
    messages    = data.get("messages", [])
    options     = data.get("options", {})
    max_tokens  = options.get("num_predict", 512)
    temperature = options.get("temperature", 0.7)
    stream      = data.get("stream", True)

    response_text = run_generation(messages, max_tokens, temperature)

    if stream:
        async def stream_response():
            words = response_text.split(" ")
            for i, word in enumerate(words):
                chunk = word + (" " if i < len(words) - 1 else "")
                yield json.dumps({
                    "model": MODEL_NAME,
                    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
                    "message": {"role": "assistant", "content": chunk},
                    "done": False
                }) + "\n"
            yield json.dumps({
                "model": MODEL_NAME,
                "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "message": {"role": "assistant", "content": ""},
                "done": True,
                "done_reason": "stop"
            }) + "\n"

        return StreamingResponse(stream_response(), media_type="application/x-ndjson")

    return JSONResponse({
        "model": MODEL_NAME,
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "message": {"role": "assistant", "content": response_text},
        "done": True,
        "done_reason": "stop"
    })


# ── Fallback generate endpoint ────────────────────────────────────────────────
@app.post("/api/generate")
async def generate(request: Request):
    data        = await request.json()
    prompt      = data.get("prompt", "")
    options     = data.get("options", {})
    max_tokens  = options.get("num_predict", 512)
    temperature = options.get("temperature", 0.7)
    stream      = data.get("stream", True)

    messages      = [{"role": "user", "content": prompt}]
    response_text = run_generation(messages, max_tokens, temperature)

    if stream:
        async def stream_response():
            for char in response_text:
                yield json.dumps({
                    "model": MODEL_NAME,
                    "response": char,
                    "done": False
                }) + "\n"
            yield json.dumps({"model": MODEL_NAME, "response": "", "done": True}) + "\n"

        return StreamingResponse(stream_response(), media_type="application/x-ndjson")

    return JSONResponse({
        "model": MODEL_NAME,
        "response": response_text,
        "done": True
    })


# ── Health check ──────────────────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"status": "ok", "model": MODEL_NAME}

In [ ]:
import os, signal, subprocess

# Kill anything on port 11434
result = subprocess.run(['fuser', '-k', '11434/tcp'], capture_output=True)
print("Cleared port 11434")

# Kill existing ngrok tunnels
ngrok.kill()

import time
time.sleep(2)  # give it a moment to release

ngrok.set_auth_token("39q2lnAykThhlBRkzZB9t79oBXx_2uxNKzyeVTAbbVCm7kwPu")  # 👈 replace this

public_url = ngrok.connect(11434, bind_tls=True)

NGROK_URL = str(public_url)
print(f"✅ Ngrok URL: {NGROK_URL}")
print(f"\n👉 Paste this into Open WebUI:")
print(f"   Admin Settings → Connections → Ollama API → {NGROK_URL}")

# Start server
config = uvicorn.Config(app, host="0.0.0.0", port=11434, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Cleared port 11434
✅ Ngrok URL: NgrokTunnel: "https://darcel-untaut-princess.ngrok-free.dev" -> "http://localhost:11434"

👉 Paste this into Open WebUI:
   Admin Settings → Connections → Ollama API → NgrokTunnel: "https://darcel-untaut-princess.ngrok-free.dev" -> "http://localhost:11434"


INFO:     Started server process [287]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:11434 (Press CTRL+C to quit)


INFO:     103.178.143.192:0 - "GET /api/tags HTTP/1.1" 200 OK
INFO:     103.178.143.192:0 - "GET /api/ps HTTP/1.1" 404 Not Found
INFO:     103.178.143.192:0 - "OPTIONS /v1/models HTTP/1.1" 404 Not Found
INFO:     103.178.143.192:0 - "GET /api/tags HTTP/1.1" 200 OK
INFO:     103.178.143.192:0 - "GET /api/ps HTTP/1.1" 404 Not Found
INFO:     103.178.143.192:0 - "OPTIONS /v1/models HTTP/1.1" 404 Not Found
INFO:     103.178.143.192:0 - "GET /api/tags HTTP/1.1" 200 OK
INFO:     103.178.143.192:0 - "GET /api/version HTTP/1.1" 404 Not Found
INFO:     103.178.143.192:0 - "GET /api/ps HTTP/1.1" 404 Not Found
INFO:     103.178.143.192:0 - "OPTIONS /v1/models HTTP/1.1" 404 Not Found
INFO:     103.178.143.192:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     103.178.143.192:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     103.178.143.192:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     103.178.143.192:0 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     103.178.143.192:0 - "GET /api/tags HTTP/1.1" 200 OK
INFO:

INFO:     Shutting down


INFO:     103.178.143.192:0 - "OPTIONS /v1/models HTTP/1.1" 404 Not Found


INFO:     Finished server process [287]
